# Early-stopping robustness check — advisor follow-up review, point 5

Every B1/B2 result in this study used ONE early-stopping patience value (8 validation events with no improvement — `PFEM_Training_Colab.ipynb`'s own `EARLY_STOP_PATIENCE=8`). This re-runs ONE case (B1 x Neo-Hookean, the representative/fastest case) with three different patience values — same seed, same architecture, same dataset, same epoch ceiling — to check whether the reported `best_val_error` is a stable property of training or an artifact of this one hyperparameter choice.

* Reuses `train_B1.py` unchanged (it already supports `--early_stop_patience`); this notebook only orchestrates three separate runs and aggregates the result.
* `BATCH_SIZE` below MUST be the case's own winning batch size from its screening protocol (`training_protocol.json`, written by Phase 1 of `PFEM_Training_Colab.ipynb`) — the cell tries to read it automatically; fill it in by hand if that file isn't found.
* Each patience value is a FULL training run to the 2000-epoch ceiling (same cost as the original production run) — budget ~3x the time of one normal B1 training run.

In [ ]:
# =====================================================================
#  CELL — Early-stopping robustness check (advisor follow-up, point 5)
#  GPU strongly preferred. Three full training runs, same seed/dataset/
#  architecture, only --early_stop_patience differs.
# =====================================================================
import os, subprocess, sys, json

def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)

from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
                    'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE - this will be slow, prefer Runtime > Change runtime type > GPU')

R = '/content/drive/MyDrive/pfem_run'
GEOMETRY, MATERIAL = 'B1', 'neo_hookean'
DATA = f'{R}/results/datasets/{GEOMETRY}_{MATERIAL}/hyperelastic_training_data_q4.npz'
OUT_DIR = f'{R}/early_stopping_robustness_{GEOMETRY}_{MATERIAL}'

protocol_path = os.path.join(R, 'results', 'training_protocol.json')
BATCH_SIZE = None
if os.path.exists(protocol_path):
    with open(protocol_path) as f:
        PROTOCOL = json.load(f)
    BATCH_SIZE = PROTOCOL.get('batch_size')
    print(f'Read batch_size={BATCH_SIZE} from {protocol_path}')
if BATCH_SIZE is None:
    raise RuntimeError(
        f'Could not read batch_size from {protocol_path} -- set BATCH_SIZE by hand '
        f"to this case's own winning batch size from screening (Phase 1 of "
        f"PFEM_Training_Colab.ipynb) before running this cell; a wrong guess would "
        f"confound this comparison with a second changed variable.")

PATIENCES = '4,8,16'   # 8 is the production value every reported result used
EPOCHS = 2000          # production protocol's own epoch ceiling (CONTINUE_EPOCHS)
VALIDATE_EVERY = 25

assert os.path.exists(DATA), f'dataset not found: {DATA}'

run([
    sys.executable, '-u', '-m', 'omar_pfem.early_stopping_robustness_check',
    '--geometry', GEOMETRY, '--material', MATERIAL,
    '--path', DATA,
    '--batch_size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--validate_every', str(VALIDATE_EVERY),
    '--patiences', PATIENCES,
    '--skip_existing',
    '--out_dir', OUT_DIR,
])

with open(os.path.join(OUT_DIR, 'early_stopping_robustness_summary.json')) as f:
    summary = json.load(f)
print('\nSummary:')
print(json.dumps(summary, indent=2))